# Business Visualisations — Roadies-CityRide

Reusable Plotly charts for key operational findings.

In [ ]:
import pandas as pd
from pathlib import Path

from roadies.ingestion.loaders import load_csv
from roadies.features.demand_supply import engineer_demand_supply_features
from roadies.features.surge import engineer_surge_features
from roadies.features.acceptance import engineer_acceptance_features
from roadies.features.cancellation import engineer_cancellation_features
from roadies.features.experience import engineer_experience_features
from roadies.features.demand_period import classify_high_demand
from roadies.database import execute_metric_query, create_database, load_dataframe
from roadies.visualization import (
    plot_city_metric,
    plot_demand_impact,
    plot_demand_supply_relationship,
    plot_city_deterioration,
    plot_temporal_pattern,
    plot_city_heatmap,
)

In [ ]:
# Load data
df = load_csv(Path('data/raw/rides.csv'))
df, _ = engineer_demand_supply_features(df)
df, _ = engineer_surge_features(df)
df, _ = engineer_acceptance_features(df)
df, _ = engineer_cancellation_features(df)
df, _ = engineer_experience_features(df)
df, _ = classify_high_demand(df)
print(f'Dataset: {len(df)} rows')

In [ ]:
# City performance comparison
city_df = df.groupby('city').agg({
    'was_accepted': 'mean',
    'rider_cancelled': 'mean',
    'wait_time_minutes': 'mean',
    'surge_multiplier': 'mean',
}).reset_index()
city_df.columns = ['city', 'acceptance_rate', 'rider_cancel_rate', 'avg_wait_time', 'avg_surge']
city_df[['acceptance_rate', 'rider_cancel_rate']] *= 100

fig = plot_city_metric(city_df, 'rider_cancel_rate', title='Rider Cancellation Rate by City')
fig.show()

In [ ]:
# High-demand impact
demand_df = df.groupby('is_high_demand').agg({
    'was_accepted': 'mean',
    'rider_cancelled': 'mean',
    'wait_time_minutes': 'mean',
    'surge_multiplier': 'mean',
}).reset_index()
demand_df['demand_period'] = demand_df['is_high_demand'].map({True: 'high', False: 'normal'})
demand_df.columns = ['is_high_demand', 'acceptance_rate', 'rider_cancel_rate', 'avg_wait_time', 'avg_surge', 'demand_period']
demand_df[['acceptance_rate', 'rider_cancel_rate']] *= 100

fig = plot_demand_impact(demand_df)
fig.show()

In [ ]:
# Demand/supply relationship
fig = plot_demand_supply_relationship(df.sample(500))
fig.show()

In [ ]:
# City heatmap
fig = plot_city_heatmap(city_df)
fig.show()